# Explicit chemical-template transfer into DockingMT

Source qualification for DockingMT #4/#5/#33 and MolSysMT #298. All molecular
operations use public MolSysMT tools. The 5X72 absent-field controls are deliberately
constructed; original SDF representation mismatches remain unassessed. The 181L
BNZ source is original heavy-only input. Successful transfer does not certify
partial charges, AutoDock typing or docking readiness.

See [the contract and reproduction record](chemical_template_consumption.md).
Execute with the qualified MolSysMT source on `PYTHONPATH` in
`molsyssuite@uibcdf_3.14`. No sibling worktree needs to be changed.

In [1]:
import json
from pathlib import Path
import runpy
import sys

assert sys.version_info[:2] == (3, 14), "Select the molsyssuite@uibcdf_3.14 kernel"
assert "molsyssuite@uibcdf_3.14" in sys.executable

import molsysmt as msm
import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'devtools/qualify_chemical_templates.py').is_file())
ns = runpy.run_path(str(root / 'devtools/qualify_chemical_templates.py'))
print(json.dumps({'python': sys.version.split()[0], 'interpreter': sys.executable,
                  'provider_import': msm.__file__,
                  'reference_source': ns['PROVIDER_REVISION']}, indent=2))

{
  "python": "3.14.7",
  "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "provider_import": "/tmp/dockingmt-template-provider-c19a47ada0c2279029abfa296cf915560610ad9a/molsysmt/__init__.py",
  "reference_source": "c19a47ada0c2279029abfa296cf915560610ad9a"
}


## Execute the bounded composition

The command verifies unchanged source/template data, atom identities, all poses,
box, time and frame/state associations. It explicitly chooses the second 5X72
frame before preparation. Reports retain original producer versions and declared
units; template hashes identify actual native snapshots separately from source
file hashes. This writes only this qualification's evidence file.

In [2]:
record = ns['qualify']()
path = root / 'devguide/validation/data/chemical_templates/cases.json'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(record, indent=2) + '\n')
print('Retained evidence:', path.relative_to(root))
print('Executed adapter versions:', record['versions'])
rows = []
for case in record['cases']:
    report = case['application_report']
    preparation = case['prepared_ligand']
    rows.append({'case': case['case'], 'status': report['status'],
                 'mapped atoms': report['source']['n_atoms'],
                 'assigned fields': len(report['assigned_fields']),
                 'preserved fields': len(report['preserved_fields']),
                 'H policy': report['coverage']['hydrogen_policy'],
                 'retained atoms': preparation['n_atoms']})
pd.DataFrame(rows)

  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



...

  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



  values = series.fillna("").astype(str).to_numpy(dtype=np.str_)



Retained evidence: devguide/validation/data/chemical_templates/cases.json
Executed adapter versions: {'rdkit': '2025.9.5', 'numpy': '2.4.6', 'pandas': '2.3.3', 'vina': '1.2.7'}


,case,status,mapped atoms,assigned fields,preserved fields,H policy,retained atoms
0,5X72 P59 permuted absent-field control,applied,39,445,0,explicit_atoms,25
1,5X72 P69 permuted absent-field control,applied,39,445,0,explicit_atoms,25
2,181L original BNZ heavy-atom source,applied,6,54,12,stored_counts,6


## Rejections remain observable

Unchanged native 5X72 SDF and adapter templates require aromatic representation
normalization that this API does not provide. Explicit charge/stereo conflicts
and a missing stored edge fail before any mutation. The command tests application
failure as well as assessment. The regression suite additionally covers incomplete
H maps, matching explicit assignments and preservation of an unselected state.

In [3]:
rows = []
for case in record['cases']:
    if 'original_sdf_assessment' in case:
        report = case['original_sdf_assessment']
        rows.append({'case': case['case'].split(' permuted')[0] + ' original SDF',
                     'status': report['status'],
                     'reasons': sorted({issue['reason_code'] for issue in report['issues']})})
for case in record['negative_controls']:
    report = case['assessment']
    rows.append({'case': case['case'], 'status': report['status'],
                 'reasons': sorted({issue['reason_code'] for issue in report['issues']})})
pd.DataFrame(rows)

,case,status,reasons
0,5X72 P59 original SDF,unassessed,[aromatic_representation_requires_normalization]
1,5X72 P69 original SDF,unassessed,[aromatic_representation_requires_normalization]
2,charge,conflict,[explicit_assignment_conflict]
3,enantiomer,conflict,[explicit_assignment_conflict]
4,missing-edge,conflict,"[isolated_connected_component_required, stored..."


## Retain the detached report and preparation limit

`prepare_ligand(application['molecular_system'])` consumes the assigned chemistry.
The provider report is currently separate from native/H5MSM and docking-result
provenance; keep it beside the workflow record. Present formal charges do not
supply partial charges. MolSysMT #298 owns normalization and native report
attachment; #300/#221/#222 own the remaining hydrogen/charge/type work.

`tests/test_chemical_template_consumption.py` exercises real Vina default
rejection, explicit exploratory execution and saved-result provisional assessment.
These scores are software controls, not experimental docking validation.

In [4]:
rows = []
for case in record['cases']:
    meta = case['prepared_ligand']['metadata']
    coverage = meta['source_chemistry']['chemical_readiness']
    assert 'docking_readiness' in coverage['unassessed_checks']
    assert meta['charge_source'] == 'zero_placeholder'
    rows.append({'case': case['case'],
                 'formal-charge coverage': coverage['fields']['formal_charge']['status'],
                 'partial-charge source': meta['charge_source'],
                 'AutoDock typing': meta['atom_type_source'],
                 'template report': case['application_report']['schema']})
pd.DataFrame(rows)

,case,formal-charge coverage,partial-charge source,AutoDock typing,template report
0,5X72 P59 permuted absent-field control,present,zero_placeholder,element_aromaticity_heuristic,molsysmt.chemical_template@1
1,5X72 P69 permuted absent-field control,present,zero_placeholder,element_aromaticity_heuristic,molsysmt.chemical_template@1
2,181L original BNZ heavy-atom source,present,zero_placeholder,element_aromaticity_heuristic,molsysmt.chemical_template@1
